## The Inspection Engine

In [2]:
import os, sys
sys.path.append("../src")
import pandas as pd
from sqlalchemy import inspect, text
from db import get_engine

engine = get_engine()
insp = inspect(engine)

## Inspecting schema

In [3]:
for col in insp.get_columns("dataset_list"):
    print(col["name"], "|", col["type"])
print("Primary key:", insp.get_pk_constraint("dataset_list"))
with engine.connect() as conn:
    print("Rows:", conn.execute(text("SELECT COUNT(*) FROM dataset_list")).scalar())

CI_Name | VARCHAR(255)
CI_Cat | VARCHAR(255)
CI_Subcat | VARCHAR(255)
WBS | VARCHAR(255)
Incident_ID | VARCHAR(255)
Status | VARCHAR(255)
Impact | VARCHAR(255)
Urgency | VARCHAR(255)
Priority | VARCHAR(255)
number_cnt | VARCHAR(255)
Category | VARCHAR(255)
KB_number | VARCHAR(255)
Alert_Status | VARCHAR(255)
No_of_Reassignments | VARCHAR(255)
Open_Time | VARCHAR(255)
Reopen_Time | VARCHAR(255)
Resolved_Time | VARCHAR(255)
Close_Time | VARCHAR(255)
Handle_Time_hrs | VARCHAR(255)
Closure_Code | VARCHAR(255)
No_of_Related_Interactions | VARCHAR(255)
Related_Interaction | VARCHAR(255)
No_of_Related_Incidents | VARCHAR(255)
No_of_Related_Changes | VARCHAR(255)
Related_Change | VARCHAR(255)
Primary key: {'name': None, 'constrained_columns': []}
Rows: 46606


## Load Dataset

In [6]:
df = pd.read_sql("SELECT * FROM dataset_list", engine)
os.makedirs("../data/raw", exist_ok=True)
df.to_csv("../data/raw/itsm_raw.csv", index=False)
print(df.shape)


(46606, 25)


## Basic Dataset Understanding

In [9]:
print("Shape of dataset:", df.shape)
print("Column names:")
print(df.columns)

print("Dataset information:")
df.info()

Shape of dataset: (46606, 25)
Column names:
Index(['CI_Name', 'CI_Cat', 'CI_Subcat', 'WBS', 'Incident_ID', 'Status',
       'Impact', 'Urgency', 'Priority', 'number_cnt', 'Category', 'KB_number',
       'Alert_Status', 'No_of_Reassignments', 'Open_Time', 'Reopen_Time',
       'Resolved_Time', 'Close_Time', 'Handle_Time_hrs', 'Closure_Code',
       'No_of_Related_Interactions', 'Related_Interaction',
       'No_of_Related_Incidents', 'No_of_Related_Changes', 'Related_Change'],
      dtype='str')
Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 46606 entries, 0 to 46605
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   CI_Name                     46606 non-null  str  
 1   CI_Cat                      46606 non-null  str  
 2   CI_Subcat                   46606 non-null  str  
 3   WBS                         46606 non-null  str  
 4   Incident_ID                 46606 non-null  st

In [11]:
df.isna().sum().sort_values(ascending=False)

CI_Name                       0
CI_Cat                        0
CI_Subcat                     0
WBS                           0
Incident_ID                   0
Status                        0
Impact                        0
Urgency                       0
Priority                      0
number_cnt                    0
Category                      0
KB_number                     0
Alert_Status                  0
No_of_Reassignments           0
Open_Time                     0
Reopen_Time                   0
Resolved_Time                 0
Close_Time                    0
Handle_Time_hrs               0
Closure_Code                  0
No_of_Related_Interactions    0
Related_Interaction           0
No_of_Related_Incidents       0
No_of_Related_Changes         0
Related_Change                0
dtype: int64

1. Every column is a string. Impact, Urgency, Priority, the No_of_... counts, Handle_Time_hrs (comma decimals) and all four date columns will need converting.
2. Zero nulls is misleading. The brief shows Reopen_Time empty in a sample row, and your earlier work found blanks and placeholders in Closure_Code, CI_Cat and CI_Subcat. Missing values here are probably stored as empty strings or placeholders like ? and NA, which isna() can’t see.
3. number_cnt isn’t in the project brief. Find out what it is. It might be a row counter, or something derived from the target.
4. Duplicates are still unchecked.

## Hidden missing values

In [12]:
placeholders = ["", "?", "NA", "N/A", "NaN", "nan", "null", "NULL", "None", "--", "-", "NS", "Unknown"]
hidden = {c: int(df[c].str.strip().isin(placeholders).sum()) for c in df.columns}
pd.Series(hidden).sort_values(ascending=False)

Related_Change                46046
No_of_Related_Changes         46046
No_of_Related_Incidents       45384
Reopen_Time                   44322
Closure_Code                   2050
Resolved_Time                  1780
Priority                       1380
Impact                         1380
No_of_Related_Interactions      114
CI_Cat                          111
CI_Subcat                       111
Handle_Time_hrs                   1
No_of_Reassignments               1
CI_Name                           0
Status                            0
WBS                               0
Incident_ID                       0
Urgency                           0
Open_Time                         0
KB_number                         0
Alert_Status                      0
number_cnt                        0
Category                          0
Close_Time                        0
Related_Interaction               0
dtype: int64

## Duplicates

In [13]:
print("Rows:", len(df))
print("Unique Incident_ID:", df["Incident_ID"].nunique())
print("Duplicate Incident_ID rows:", df["Incident_ID"].duplicated().sum())
print("Fully duplicated rows:", df.duplicated().sum())

Rows: 46606
Unique Incident_ID: 46606
Duplicate Incident_ID rows: 0
Fully duplicated rows: 0


## Cardinality and the unclear column

In [14]:
df.nunique().sort_values()
df["number_cnt"].value_counts().head(10)

number_cnt
0.601292279    1
0.415049969    1
0.517551335    1
0.642927218    1
0.345258343    1
0.006675648    1
0.19290712     1
0.996467545    1
0.269573825    1
0.306670312    1
Name: count, dtype: int64

## Small categorical columns and the priority matrix

In [17]:
for c in ["Status", "Impact", "Urgency", "Priority", "Category", "Alert_Status"]:
    print(df[c].value_counts().head(10), "\n")

Status
Closed              46597
Work in progress        9
Name: count, dtype: int64 

Impact
4     22556
5     16741
3      5234
NS     1380
2       692
1         3
Name: count, dtype: int64 

Urgency
4               22588
5               16779
3                6536
2                 696
1                   6
5 - Very Low        1
Name: count, dtype: int64 

Priority
4     22717
5     16486
3      5323
NA     1380
2       697
1         3
Name: count, dtype: int64 

Category
incident                   37748
request for information     8846
complaint                     11
request for change             1
Name: count, dtype: int64 

Alert_Status
closed    46606
Name: count, dtype: int64 



In [16]:
pd.crosstab([df["Impact"], df["Urgency"]], df["Priority"])

Priority             1    2     3      4      5    NA
Impact Urgency                                       
1      1             3    0     0      0      0     0
2      2             0  689     0      0      0     0
       3             0    3     0      0      0     0
3      1             0    2     0      0      0     0
       2             0    3     0      0      0     0
       3             0    0  5077      0      0     0
       4             0    0   144      0      0     0
       5             0    0     0      8      0     0
4      2             0    0     3      0      0     0
       3             0    0    97      0      0     0
       4             0    0     0  22170      0     0
       5             0    0     0    286      0     0
5      1             0    0     1      0      0     0
       2             0    0     1      0      0     0
       3             0    0     0     11      0     0
       4             0    0     0    242      0     0
       5             0    0     0      0  16485     0
       5 - Very Low  0    0     0      0      1     0
NS     3             0    0     0      0      0  1348
       4             0    0     0      0      0    32

## Handle time format

In [18]:
print("Handle_Time_hrs with comma:", df["Handle_Time_hrs"].str.contains(",").sum())

Handle_Time_hrs with comma: 46349


## date-format check

In [23]:
print(df["number_cnt"].astype(float).describe())
print(df["Closure_Code"].value_counts().head(8))

parts = df["Open_Time"].str.extract(r"^(\d{1,2})-(\d{1,2})-(\d{4})").apply(pd.to_numeric, errors="coerce")
print("Rows not matching dd-dd-yyyy:", parts[0].isna().sum())
print("First part max:", parts[0].max(), "| Second part max:", parts[1].max())
print("Year range:", parts[2].min(), "to", parts[2].max())



count    46606.000000
mean         0.499658
std          0.288634
min          0.000023
25%          0.248213
50%          0.500269
75%          0.749094
max          0.999997
Name: number_cnt, dtype: float64
Closure_Code
Other                           16470
Software                        13027
User error                       3554
No error - works as designed     3530
Hardware                         2999
Data                             2209
Unknown                          1590
Operator error                   1539
Name: count, dtype: int64
Rows not matching dd-dd-yyyy: 0
First part max: 31 | Second part max: 12
Year range: 2012 to 2014


## Check the other time columns

In [22]:
for c in ["Resolved_Time", "Close_Time", "Reopen_Time"]:
    s = df[c].str.strip()
    s = s[s != ""]
    print(c, "| non-blank:", len(s), "| matching pattern:", s.str.match(r"^\d{1,2}-\d{1,2}-\d{4} \d{1,2}:\d{2}").sum())

Resolved_Time | non-blank: 44826 | matching pattern: 44826
Close_Time | non-blank: 46606 | matching pattern: 46606
Reopen_Time | non-blank: 2284 | matching pattern: 2284


Findings that matter more

* Priority is a lookup of Impact and Urgency. Every row of your crosstab has exactly one non-zero cell, so each Impact and Urgency pair maps to one priority. If you give a model both columns, it just relearns the matrix and the scores will look great for no real reason. Decide with your mentor between predicting priority from other fields (CI_Cat, CI_Subcat, Category, WBS, open time) or predicting Impact and Urgency themselves. I’d report the model both with and without Impact and Urgency, so the contrast is visible.
* Severe imbalance. Only 3 tickets are P1 and 697 are P2, so P1/P2 is about 1.5% of rows. A five-class model can’t learn P1 from 3 examples, so I’d merge P1 and P2 into “High” for the main model.
* number_cnt looks like noise. It holds random-looking decimals, all unique. It isn’t in the brief, so I’d drop it after one check below.
* Alert_Status is constant (“closed” in every row), so drop it. Status is 99.98% “Closed” and only known after handling, so drop it too.
* Label inconsistency: Urgency has one 5 - Very Low that should become 5.
* Track 4 hint: only 560 tickets have a related change, and Category has just 1 “request for change”. Ask your mentor which one defines the RFC target.